# 02 - IO-VNBD Data Preprocessing & Feature Engineering Validation

This notebook validates the preprocessing pipeline for **SIH26168 Intelligent Dead Reckoning**.
It loads raw smartphone sequences from `dataset/raw/IO-VNBD-master/`, normalizes column names, parses timestamps, applies a 4th-order zero-phase Butterworth low-pass filter to IMU channels, computes physical derived features, and inspects quality-control figures.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

# Add project root
sys.path.insert(0, str(Path.cwd().parent))

from ml.preprocessing import (
    IOVNBDLoader,
    TimestampProcessor,
    SensorCleaner,
    SensorFilter,
    FeatureEngineer,
    PreprocessingPipelineRunner,
)

## 1. Discover Smartphone Sequences

In [ ]:
loader = IOVNBDLoader(raw_dataset_dir="../dataset/raw/IO-VNBD-master")
seq_paths = loader.discover_smartphone_sequences()
print(f"Discovered {len(seq_paths)} smartphone sequences.")
for p in seq_paths[:5]:
    print(f" - {p.name}")

## 2. Load & Preprocess Sample Sequence

In [ ]:
sample_path = seq_paths[0]
seq_data = loader.load_sequence(sample_path)

ts_proc = TimestampProcessor()
df_ts, ts_stats = ts_proc.process_sequence_timestamps(seq_data.clean_df)

cleaner = SensorCleaner()
df_clean, clean_stats = cleaner.clean_sequence_sensors(df_ts)

filt = SensorFilter()
df_filt = filt.filter_sequence(df_clean)

eng = FeatureEngineer()
df_final = eng.engineer_features(df_filt)

print(f"Sequence ID: {seq_data.sequence_id}")
print(f"Final Processed Shape: {df_final.shape}")
print(f"Columns: {list(df_final.columns)[:10]}...")

## 3. Visualize Quality-Control Plots

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df_final["timestamp_sec"], df_final["accel_raw_z_ms2"], "b-", alpha=0.4, label="Raw Accel Z")
plt.plot(df_final["timestamp_sec"], df_final["accel_filtered_z_ms2"], "r-", linewidth=1.5, label="Filtered Accel Z (3Hz Lowpass)")
plt.title(f"Raw vs Filtered Accelerometer Z - {seq_data.sequence_id}")
plt.xlabel("Time (sec)")
plt.ylabel("m/s²")
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend()
plt.show()